# Universal Fraud Detection — Training & Deployment (Final)
- Reads from: s3://databucket-fraud-detection/output_data/universal-features/
- Trains XGBoost on 3 combined datasets
- Deploys using SKLearnModel + inference.py (same as cc_fraud_training_v2)
- New endpoint: universal-fraud-endpoint on ml.t2.medium

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q xgboost scikit-learn pyarrow s3fs pandas numpy boto3 sagemaker
print('Done.')

In [ ]:
# ── 2. Imports ───────────────────────────────────────────────────────────
import os, json, tarfile, joblib, pathlib, botocore, time
import boto3
import numpy as np
import pandas as pd
import s3fs
import sagemaker
from sagemaker import get_execution_role
from sagemaker.sklearn.model import SKLearnModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, average_precision_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

print('All imports OK')

In [ ]:
# ── 3. Config ────────────────────────────────────────────────────────────
REGION         = 'us-east-1'
PRIMARY_BUCKET = 'databucket-fraud-detection'
DR_BUCKET      = 'fraud-detection-dr'

INPUT_S3      = 's3://databucket-fraud-detection/output_data/universal-features/'
ENDPOINT_NAME = 'universal-fraud-endpoint'   # new endpoint
TARGET_COL    = 'label'
DROP_COLS     = ['source']

# Exact EMR feature schema
EMR_FEATURE_COLS = [
    'log_amount', 'amount_z', 'is_high_amount',
    'hour', 'hour_sin', 'hour_cos', 'is_night',
    'day_of_week', 'is_weekend', 'txn_month',
    'user_mean_amount', 'user_std_amount',
    'user_txn_count', 'user_amount_z',
    'txn_count_5m',
    'pca_l2', 'pca_max_abs', 'pca_mean'
]

sess      = sagemaker.Session()
role      = get_execution_role()
s3c       = boto3.client('s3', region_name=REGION)
sm_client = boto3.client('sagemaker', region_name=REGION)

print(f'Role     : {role}')
print(f'Input    : {INPUT_S3}')
print(f'Endpoint : {ENDPOINT_NAME}')

In [ ]:
# ── 4. Load EMR Output (1.5M rows sample) ────────────────────────────────
fs    = s3fs.S3FileSystem(anon=False)
files = fs.glob(f'{PRIMARY_BUCKET}/output_data/universal-features/**/*.parquet')

if not files:
    raise RuntimeError('No parquet files found. EMR job may not have completed.')

print(f'Found {len(files)} parquet files')

dfs        = []
total_rows = 0
TARGET_ROWS = 1_500_000

for f in files:
    chunk = pd.read_parquet(f's3://{f}', storage_options={'anon': False})
    dfs.append(chunk)
    total_rows += len(chunk)
    if total_rows >= TARGET_ROWS:
        break

df = pd.concat(dfs, ignore_index=True).iloc[:TARGET_ROWS]

print(f'Shape  : {df.shape}')
print(f'Memory : {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
print(f'Sources: {df["source"].value_counts().to_dict()}')
df.head()

In [ ]:
# ── 5. EDA ────────────────────────────────────────────────────────────────
print('=== Sources ===')
print(df['source'].value_counts())

print('\n=== Label Distribution ===')
print(df[TARGET_COL].value_counts())
print(f'Fraud rate: {df[TARGET_COL].mean():.4%}')

print('\n=== Schema Check ===')
missing = [c for c in EMR_FEATURE_COLS if c not in df.columns]
print(f'Missing cols: {missing if missing else "None — schema OK"}')

print('\n=== Missing Values ===')
mv = df.isnull().sum()
print(mv[mv > 0] if mv.any() else 'No missing values')

In [ ]:
# ── 6. Prepare X / y ──────────────────────────────────────────────────────
drop = [c for c in DROP_COLS if c in df.columns]
X    = df.drop(columns=[TARGET_COL] + drop)
y    = df[TARGET_COL].astype(int)

X = X.reindex(columns=EMR_FEATURE_COLS, fill_value=0)
X = X.replace([np.inf, -np.inf], np.nan)

FEATURE_COLS = list(X.columns)

print(f'Features : {X.shape[1]}')
print(f'Samples  : {len(y)}')
print(f'Fraud    : {y.sum()} ({y.mean():.2%})')

In [ ]:
# ── 7. Train / Test Split ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train : {X_train.shape} | Fraud: {y_train.sum()}')
print(f'Test  : {X_test.shape}  | Fraud: {y_test.sum()}')

In [ ]:
# ── 8. Preprocessing ───────────────────────────────────────────────────────
preprocess = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler(with_mean=False)),
])

X_train_p = preprocess.fit_transform(X_train)
X_test_p  = preprocess.transform(X_test)
print(f'Train preprocessed: {X_train_p.shape}')

In [ ]:
# ── 9. Train XGBoost ───────────────────────────────────────────────────────
scale_pos = int((y_train==0).sum() / max((y_train==1).sum(), 1))
sw        = compute_sample_weight('balanced', y_train)

model = xgb.XGBClassifier(
    n_estimators          = 300,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    scale_pos_weight      = scale_pos,
    eval_metric           = 'aucpr',
    random_state          = 42,
    tree_method           = 'hist',
    early_stopping_rounds = 30,
)

model.fit(
    X_train_p, y_train,
    sample_weight = sw,
    eval_set      = [(X_test_p, y_test)],
    verbose       = 50
)
print('Training complete.')

In [ ]:
# ── 10. Evaluate ───────────────────────────────────────────────────────────
y_pred  = model.predict(X_test_p)
y_proba = model.predict_proba(X_test_p)[:, 1]

metrics = {
    'roc_auc' : round(roc_auc_score(y_test, y_proba), 4),
    'pr_auc'  : round(average_precision_score(y_test, y_proba), 4),
    'n_test'  : len(y_test),
    'n_fraud' : int(y_test.sum()),
    'features': len(FEATURE_COLS),
    'endpoint': ENDPOINT_NAME
}

print('=== Metrics ===')
print(json.dumps(metrics, indent=2))
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Legit','Fraud']))
print('\n=== Confusion Matrix ===')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# ── 11. Save model as joblib (same as cc_fraud_training_v2) ───────────────
local_dir = pathlib.Path('/tmp/universal_model')
local_dir.mkdir(parents=True, exist_ok=True)

# Save as joblib — required for SKLearnModel inference
joblib.dump(model,        local_dir / 'xgb_model.joblib')
joblib.dump(preprocess,   local_dir / 'preprocessor.joblib')
joblib.dump(FEATURE_COLS, local_dir / 'feature_names.joblib')

with open(local_dir / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

tar_path = '/tmp/universal_model.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    for fp in local_dir.iterdir():
        tar.add(fp, arcname=fp.name)

print(f'Packaged: {[fp.name for fp in local_dir.iterdir()]}')

In [ ]:
# ── 12. Upload to PRIMARY S3 ──────────────────────────────────────────────
def s3_upload(local_path, bucket, key):
    s3c.upload_file(local_path, bucket, key)
    print(f'Uploaded → s3://{bucket}/{key}')

s3_upload(tar_path,
          PRIMARY_BUCKET, 'models/universal-fraud-xgb/model.tar.gz')
s3_upload(str(local_dir/'metrics.json'),
          PRIMARY_BUCKET, 'results/universal-fraud-xgb/metrics.json')
s3_upload(str(local_dir/'feature_names.joblib'),
          PRIMARY_BUCKET, 'results/universal-fraud-xgb/feature_names.joblib')
print('Primary upload done.')

In [ ]:
# ── 13. Backup to DR S3 ────────────────────────────────────────────────────
try:
    s3_upload(tar_path,
              DR_BUCKET, 'models/universal-fraud-xgb/model.tar.gz')
    s3_upload(str(local_dir/'metrics.json'),
              DR_BUCKET, 'results/universal-fraud-xgb/metrics.json')
    print('DR backup complete.')
except Exception as e:
    print(f'DR backup failed: {e}')

In [ ]:
# ── 14. Write inference script (same pattern as cc_fraud_training_v2) ─────
inference_script = '''
import joblib, os, json
import numpy as np
import pandas as pd

def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, 'xgb_model.joblib'))
    prep  = joblib.load(os.path.join(model_dir, 'preprocessor.joblib'))
    cols  = joblib.load(os.path.join(model_dir, 'feature_names.joblib'))
    return {'model': model, 'prep': prep, 'cols': cols}

def input_fn(request_body, content_type):
    if content_type == 'application/json':
        return pd.DataFrame(json.loads(request_body))
    raise ValueError(f'Unsupported content type: {content_type}')

def predict_fn(input_data, bundle):
    X = input_data.reindex(columns=bundle["cols"], fill_value=0)
    X = bundle["prep"].transform(X)
    proba = bundle["model"].predict_proba(X)[:, 1]
    pred  = bundle["model"].predict(X)
    return {'predictions': pred.tolist(), 'probabilities': proba.tolist()}

def output_fn(prediction, accept):
    return json.dumps(prediction), 'application/json'
'''

os.makedirs('/tmp/inference', exist_ok=True)
with open('/tmp/inference/inference.py', 'w') as f:
    f.write(inference_script)

print('Inference script written.')

In [ ]:
# ── 15. Deploy NEW Endpoint using SKLearnModel (same as cc_fraud_training_v2)
model_uri = f's3://{PRIMARY_BUCKET}/models/universal-fraud-xgb/model.tar.gz'

sklearn_model = SKLearnModel(
    model_data        = model_uri,
    role              = role,
    entry_point       = '/tmp/inference/inference.py',
    framework_version = '1.2-1',
    py_version        = 'py3',
)

# Clean up if endpoint name already exists
for fn, label in [
    (lambda: sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME), 'endpoint'),
    (lambda: sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME), 'config'),
]:
    try:
        fn()
        print(f'Deleted old {label}')
        time.sleep(15)
    except botocore.exceptions.ClientError:
        print(f'No existing {label} — skipping')

print(f'\nDeploying {ENDPOINT_NAME} ...')
predictor = sklearn_model.deploy(
    initial_instance_count = 1,
    instance_type          = 'ml.t2.medium',
    endpoint_name          = ENDPOINT_NAME,
)
print(f'Endpoint live: {ENDPOINT_NAME}')

In [ ]:
# ── 16. Test Endpoint ──────────────────────────────────────────────────────
runtime = boto3.client('sagemaker-runtime', region_name=REGION)

# Send as JSON dict (SKLearnModel expects application/json)
sample = X_test.iloc[:5].to_dict(orient='records')

response = runtime.invoke_endpoint(
    EndpointName = ENDPOINT_NAME,
    ContentType  = 'application/json',
    Body         = json.dumps(sample)
)
result = json.loads(response['Body'].read())

print('Endpoint test OK')
print(f'Probabilities : {[round(p,4) for p in result["probabilities"]]}')
print(f'Predictions   : {result["predictions"]}')
print(f'Actual labels : {y_test.iloc[:5].tolist()}')

In [ ]:
# ── 17. Universal Predict Function ─────────────────────────────────────────
# Uses application/json — aligned to SKLearnModel inference.py

def predict_any_csv(csv_path_or_s3,
                    amount_col,
                    time_col        = None,
                    time_is_numeric = False,
                    user_col        = None,
                    threshold       = 0.5):
    # Load
    if csv_path_or_s3.startswith('s3://'):
        raw_df = pd.read_csv(csv_path_or_s3,
                             storage_options={'anon': False})
    else:
        raw_df = pd.read_csv(csv_path_or_s3)
    print(f'Loaded : {raw_df.shape}')

    feat = pd.DataFrame(index=raw_df.index)

    # Amount features
    amt = pd.to_numeric(raw_df[amount_col], errors='coerce').fillna(0)
    feat['log_amount']     = np.log1p(amt)
    feat['amount_z']       = (amt - amt.mean()) / (amt.std() or 1)
    feat['is_high_amount'] = (amt > amt.quantile(0.95)).astype(int)

    # Time features
    if time_col and time_col in raw_df.columns:
        if time_is_numeric:
            hour = (pd.to_numeric(raw_df[time_col]) / 3600) % 24
            feat['day_of_week'] = 0
            feat['is_weekend']  = 0
            feat['txn_month']   = 0
        else:
            dt   = pd.to_datetime(raw_df[time_col], errors='coerce')
            hour = dt.dt.hour
            feat['day_of_week'] = dt.dt.dayofweek
            feat['is_weekend']  = (dt.dt.dayofweek >= 5).astype(int)
            feat['txn_month']   = dt.dt.month
        feat['hour']     = hour
        feat['hour_sin'] = np.sin(2 * np.pi * hour / 24)
        feat['hour_cos'] = np.cos(2 * np.pi * hour / 24)
        feat['is_night'] = ((hour >= 22) | (hour <= 6)).astype(int)
    else:
        for c in ['hour','hour_sin','hour_cos','is_night',
                  'day_of_week','is_weekend','txn_month']:
            feat[c] = 0

    # User features
    if user_col and user_col in raw_df.columns:
        us = raw_df.groupby(user_col)[amount_col].agg(
            user_mean_amount='mean',
            user_std_amount='std',
            user_txn_count='count'
        ).reset_index()
        m = raw_df[[user_col, amount_col]].merge(us, on=user_col, how='left')
        feat['user_mean_amount'] = m['user_mean_amount'].values
        feat['user_std_amount']  = m['user_std_amount'].fillna(1).values
        feat['user_txn_count']   = m['user_txn_count'].values
        feat['user_amount_z']    = (
            (amt - m['user_mean_amount'].values) /
            m['user_std_amount'].fillna(1).values
        )
    else:
        feat['user_mean_amount'] = 0.0
        feat['user_std_amount']  = 0.0
        feat['user_txn_count']   = 0
        feat['user_amount_z']    = 0.0

    feat['txn_count_5m'] = 0

    # PCA features
    v_cols = [c for c in raw_df.columns
              if c.startswith('V') and c[1:].isdigit()]
    if v_cols:
        v = raw_df[v_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
        feat['pca_l2']      = np.sqrt((v**2).sum(axis=1))
        feat['pca_max_abs'] = v.abs().max(axis=1)
        feat['pca_mean']    = v.mean(axis=1)
    else:
        feat['pca_l2']      = 0.0
        feat['pca_max_abs'] = 0.0
        feat['pca_mean']    = 0.0

    # Align to training schema
    feat = feat.reindex(columns=FEATURE_COLS, fill_value=0).fillna(0)

    # Batch inference — JSON format for SKLearnModel
    runtime    = boto3.client('sagemaker-runtime', region_name=REGION)
    all_probas = []
    all_preds  = []

    for i in range(0, len(feat), 100):
        batch   = feat.iloc[i:i+100].to_dict(orient='records')
        resp    = runtime.invoke_endpoint(
            EndpointName = ENDPOINT_NAME,
            ContentType  = 'application/json',
            Body         = json.dumps(batch)
        )
        result = json.loads(resp['Body'].read())
        all_probas.extend(result['probabilities'])
        all_preds.extend(result['predictions'])

    raw_df['fraud_probability'] = all_probas
    raw_df['fraud_predicted']   = all_preds

    print(f'Total         : {len(raw_df)}')
    print(f'Fraud detected: {sum(all_preds)}')
    print(f'Fraud rate    : {sum(all_preds)/len(all_preds):.2%}')
    return raw_df

print('predict_any_csv() ready.')

In [ ]:
# ── 18. Predict — creditcard.csv ───────────────────────────────────────────
r1 = predict_any_csv(
    's3://databucket-fraud-detection/input_data/creditcard.csv',
    amount_col      = 'Amount',
    time_col        = 'Time',
    time_is_numeric = True
)
r1[['Amount','Class','fraud_probability','fraud_predicted']].head(10)

In [ ]:
# ── 19. Predict — financial_fraud_detection_dataset.csv ────────────────────
r2 = predict_any_csv(
    's3://databucket-fraud-detection/input_data/financial_fraud_detection_dataset.csv',
    amount_col = 'amount',
    time_col   = 'timestamp',
    user_col   = 'sender_account'
)
r2[['amount','is_fraud','fraud_probability','fraud_predicted']].head(10)

In [ ]:
# ── 20. Predict — credit_card_fraud_dataset.csv ────────────────────────────
r3 = predict_any_csv(
    's3://databucket-fraud-detection/input_data/credit_card_fraud_dataset.csv',
    amount_col = 'Amount',
    time_col   = 'TransactionDate',
    user_col   = 'MerchantID'
)
r3[['Amount','IsFraud','fraud_probability','fraud_predicted']].head(10)

In [ ]:
# ── 21. Predict — mixed_fraud_test_dataset_1M.csv (NEW DATASET) ────────────
r4 = predict_any_csv(
    's3://databucket-fraud-detection/input_data/mixed_fraud_test_dataset_1M.csv',
    amount_col = 'Amount',
    time_col   = 'timestamp',
    user_col   = 'sender_account'
)
print('\nAccuracy on new 1M dataset:')
print(classification_report(
    r4['IsFraud'],
    r4['fraud_predicted'],
    target_names=['Legit','Fraud']
))

In [ ]:
# ── 22. Save all predictions to S3 ────────────────────────────────────────
for name, result in [
    ('creditcard',        r1),
    ('financial',         r2),
    ('credit_card_fraud', r3),
    ('mixed_1M',          r4)
]:
    tmp = f'/tmp/{name}_predictions.csv'
    result.to_csv(tmp, index=False)
    s3_upload(tmp, PRIMARY_BUCKET,
              f'results/universal-fraud-xgb/predictions/{name}_predictions.csv')

print('All predictions saved to S3.')

In [ ]:
# ── 23. Final Summary ─────────────────────────────────────────────────────
print('='*65)
print('  UNIVERSAL FRAUD DETECTION — COMPLETE')
print('='*65)
print(f'  EMR Input       : {INPUT_S3}')
print(f'  Samples trained : {len(y):,}')
print(f'  Features        : {len(FEATURE_COLS)}')
print(f'  ROC-AUC         : {metrics["roc_auc"]}')
print(f'  PR-AUC          : {metrics["pr_auc"]}')
print(f'  NEW Endpoint    : {ENDPOINT_NAME} (ml.t2.medium, SKLearnModel)')
print(f'  OLD Endpoint    : fraud-detection-endpoint (untouched)')
print(f'  Model Primary   : s3://{PRIMARY_BUCKET}/models/universal-fraud-xgb/')
print(f'  Model DR Backup : s3://{DR_BUCKET}/models/universal-fraud-xgb/')
print(f'  Predictions     : s3://{PRIMARY_BUCKET}/results/universal-fraud-xgb/predictions/')
print('='*65)
print('  Works on ANY dataset with an Amount column!')
print('='*65)